# 第9课：比较Fixed 0/25/50/75/100% Hedge

本课把第8课的Fixed 50%公式写成一个可重复使用的函数，然后在**完全相同的10,000个情景**下比较五种固定套保比例。

本课只比较fixed strategies；adaptive July strategy留到下一课。

## 0. 为什么必须使用同一批情景？

如果Fixed 25%和Fixed 75%分别使用不同的天气、产量和价格随机数，利润差异可能只是随机抽样造成的。

本项目使用common random numbers：每个`scenario_id`下，五个策略面对完全相同的yield、July futures、Harvest futures、basis和cash price。唯一改变的是套保规则。

因此：

$$ProfitDifference_i=Profit_{StrategyA,i}-Profit_{StrategyB,i}$$

可以被解释为策略差异，而不是情景差异。

## 1. 先锁定选择标准

教授提醒：expected profit、variability和downside risk可能支持不同策略。因此我们不能看完结果后才决定用哪个指标。

本项目的最终选择规则为：

1. **主要目标**：在合格策略中，选择CVaR 5%最高的策略，也就是最差5%情景的平均利润最不差；
2. **Expected-profit safeguard**：策略平均利润不能比unhedged mean低超过$5\%\times|UnhedgedMean|$；
3. **Over-hedging safeguard**：over-hedging probability不能超过10%；
4. Standard deviation、P5、loss probability和margin-call proxy作为辅助报告指标。

本课得到的只能是“fixed strategies中的暂时最佳”，因为adaptive策略尚未加入。

## 2. 导入工具并锁定参数

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
FARM_ACRES = 1_000
CONTRACT_SIZE_BUSHELS = 5_000
FIXED_HEDGE_RATIOS = [0.00, 0.25, 0.50, 0.75, 1.00]

TRANSACTION_COST_PER_CONTRACT_SIDE = 25.00
INITIAL_MARGIN_PER_CONTRACT = 2_500.00
MARGIN_FINANCING_RATE_ANNUAL = 0.06
DAYS_PRESEASON_TO_JULY = 136
DAYS_JULY_TO_HARVEST = 108
MARGIN_LIQUIDITY_RESERVE = 50_000.00

LOWER_TAIL_PROBABILITY = 0.05
MAX_MEAN_PROFIT_SHORTFALL_FRACTION = 0.05
MAX_OVERHEDGE_PROBABILITY = 0.10
Z_95 = 1.96

print('Fixed hedge ratios:', FIXED_HEDGE_RATIOS)
print('Monte Carlo scenarios per strategy:', N_SIMULATIONS)

## 3. 读取第7课共同情景

本课从第7课的unhedged baseline开始，确保五种策略都使用同一份原始情景。

In [ ]:
candidate_paths = [
    Path.cwd() / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
    Path.cwd().parent / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
]

lesson7_path = next((p for p in candidate_paths if p.exists()), None)
if lesson7_path is None:
    raise FileNotFoundError(
        '没有找到第7课CSV。请先运行Lesson_07 Notebook，并保留lesson_07_outputs文件夹。'
    )

scenarios = pd.read_csv(lesson7_path)

required_columns = [
    'scenario_id',
    'preseason_expected_yield_bu_per_acre',
    'actual_production_bushels',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'cash_revenue_usd',
    'production_cost_usd',
]
missing = [c for c in required_columns if c not in scenarios.columns]

assert not missing, f'缺少列: {missing}'
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[required_columns].isna().sum().sum() == 0

print('共同情景读取与检查通过:', lesson7_path)

## 4. 建立Nearest-Whole-Contract函数

$$Contracts=\left\lfloor\frac{HedgedBushels}{5{,}000}+0.5\right\rfloor$$

In [ ]:
def rounded_contracts(quantity_bushels, contract_size=CONTRACT_SIZE_BUSHELS):
    quantity_bushels = np.asarray(quantity_bushels, dtype=float)
    return np.floor(np.maximum(quantity_bushels, 0.0) / contract_size + 0.5).astype(int)

test_quantities = np.array([0, 52_562.64, 105_125.29, 157_687.93, 210_250.57])
print(pd.DataFrame({
    'Target bushels': test_quantities,
    'Rounded contracts': rounded_contracts(test_quantities),
}).to_string(index=False))

## 5. 建立Fixed Strategy计算函数

每个比例都使用同一套公式：

$$InitialPnL=N_0\times5{,}000\times(F_0-F_H)$$

$$JulyAdjustmentPnL=0\times5{,}000\times(F_J-F_H)=0$$

$$Profit=CashRevenue+InitialPnL-TransactionCost-MarginCost-ProductionCost$$

函数输出long-format结果：每一行是一个`scenario × strategy`组合。

In [ ]:
def evaluate_fixed_strategy(hedge_ratio, common_scenarios):
    n = len(common_scenarios)
    strategy_name = f'Fixed {int(round(hedge_ratio * 100))}%'

    expected_production = (
        common_scenarios['preseason_expected_yield_bu_per_acre'].to_numpy()
        * FARM_ACRES
    )
    initial_contracts = rounded_contracts(hedge_ratio * expected_production)
    july_adjustment_contracts = np.zeros(n, dtype=int)
    final_contracts = initial_contracts + july_adjustment_contracts

    f0 = common_scenarios['preseason_futures_usd_per_bushel'].to_numpy()
    f_july = common_scenarios['july_futures_usd_per_bushel'].to_numpy()
    f_harvest = common_scenarios['harvest_futures_usd_per_bushel'].to_numpy()

    initial_futures_pnl = (
        initial_contracts * CONTRACT_SIZE_BUSHELS * (f0 - f_harvest)
    )
    july_adjustment_pnl = (
        july_adjustment_contracts
        * CONTRACT_SIZE_BUSHELS
        * (f_july - f_harvest)
    )
    total_futures_pnl = initial_futures_pnl + july_adjustment_pnl

    transaction_sides = (
        np.abs(initial_contracts)
        + np.abs(july_adjustment_contracts)
        + np.abs(final_contracts)
    )
    transaction_cost = transaction_sides * TRANSACTION_COST_PER_CONTRACT_SIDE

    margin_financing_cost = (
        INITIAL_MARGIN_PER_CONTRACT
        * MARGIN_FINANCING_RATE_ANNUAL
        * (
            np.abs(initial_contracts) * DAYS_PRESEASON_TO_JULY / 365.0
            + np.abs(final_contracts) * DAYS_JULY_TO_HARVEST / 365.0
        )
    )

    cash_revenue = common_scenarios['cash_revenue_usd'].to_numpy()
    gross_revenue_after_hedge = (
        cash_revenue
        + total_futures_pnl
        - transaction_cost
        - margin_financing_cost
    )
    profit = (
        gross_revenue_after_hedge
        - common_scenarios['production_cost_usd'].to_numpy()
    )

    actual_production = common_scenarios['actual_production_bushels'].to_numpy()
    final_hedged_bushels = final_contracts * CONTRACT_SIZE_BUSHELS
    overhedged = final_hedged_bushels > actual_production

    july_mtm = initial_contracts * CONTRACT_SIZE_BUSHELS * (f0 - f_july)
    harvest_segment_mtm = final_contracts * CONTRACT_SIZE_BUSHELS * (f_july - f_harvest)
    margin_call_proxy = (
        ((-july_mtm) > MARGIN_LIQUIDITY_RESERVE)
        | ((-harvest_segment_mtm) > MARGIN_LIQUIDITY_RESERVE)
    )

    return pd.DataFrame({
        'scenario_id': common_scenarios['scenario_id'].to_numpy(),
        'strategy': strategy_name,
        'initial_hedge_ratio': hedge_ratio,
        'july_target_hedge_ratio': hedge_ratio,
        'initial_contracts': initial_contracts,
        'july_adjustment_contracts': july_adjustment_contracts,
        'final_contracts': final_contracts,
        'actual_production_bushels': actual_production,
        'final_hedged_bushels': final_hedged_bushels,
        'cash_revenue_usd': cash_revenue,
        'initial_futures_pnl_usd': initial_futures_pnl,
        'july_adjustment_pnl_usd': july_adjustment_pnl,
        'total_futures_pnl_usd': total_futures_pnl,
        'transaction_cost_usd': transaction_cost,
        'margin_financing_cost_usd': margin_financing_cost,
        'gross_revenue_after_hedge_usd': gross_revenue_after_hedge,
        'profit_usd': profit,
        'profit_usd_per_acre': profit / FARM_ACRES,
        'overhedged': overhedged,
        'margin_call_proxy': margin_call_proxy,
    })

## 6. 运行五种固定策略

In [ ]:
fixed_results = pd.concat(
    [evaluate_fixed_strategy(ratio, scenarios) for ratio in FIXED_HEDGE_RATIOS],
    ignore_index=True,
)

print('结果行数:', len(fixed_results))
print('策略:', fixed_results['strategy'].unique().tolist())
print(fixed_results.head(8).round(3).to_string(index=False))

## 7. 检查合约数与实际套保比例

期货只能交易整数合约，所以实际比例不会恰好等于目标比例。

In [ ]:
position_table = []
expected_production_2026 = (
    scenarios['preseason_expected_yield_bu_per_acre'].iloc[0] * FARM_ACRES
)

for strategy, group in fixed_results.groupby('strategy', sort=False):
    contracts = int(group['initial_contracts'].iloc[0])
    hedged_bushels = contracts * CONTRACT_SIZE_BUSHELS
    position_table.append({
        'Strategy': strategy,
        'Target Ratio': group['initial_hedge_ratio'].iloc[0],
        'Initial Contracts': contracts,
        'Hedged Bushels': hedged_bushels,
        'Actual Ratio After Rounding': hedged_bushels / expected_production_2026,
        'July Adjustment': int(group['july_adjustment_contracts'].iloc[0]),
    })

position_table = pd.DataFrame(position_table)
print(position_table.round(4).to_string(index=False))

## 8. 验证两段P&L没有混合

所有fixed strategies的July adjustment必须为0，所以July adjustment P&L也必须为0。Initial P&L必须只使用$F_0$和$F_H$。

In [ ]:
assert (fixed_results['july_adjustment_contracts'] == 0).all()
assert np.allclose(fixed_results['july_adjustment_pnl_usd'], 0.0)
assert np.allclose(
    fixed_results['total_futures_pnl_usd'],
    fixed_results['initial_futures_pnl_usd'],
)

for strategy, group in fixed_results.groupby('strategy', sort=False):
    expected_pnl = (
        group['initial_contracts'].to_numpy()
        * CONTRACT_SIZE_BUSHELS
        * (
            scenarios['preseason_futures_usd_per_bushel'].to_numpy()
            - scenarios['harvest_futures_usd_per_bushel'].to_numpy()
        )
    )
    assert np.allclose(group['initial_futures_pnl_usd'], expected_pnl)

print('两段P&L检查通过。')

## 9. 建立策略汇总函数

每个策略都报告教授关心的三类结果：

- Expected profit；
- Variability；
- Downside risk。

同时报告实施成本、over-hedging和margin liquidity proxy。

In [ ]:
def summarize_strategy(group):
    profit = group['profit_usd_per_acre']
    p5 = profit.quantile(LOWER_TAIL_PROBABILITY)
    cvar5 = profit[profit <= p5].mean()
    standard_error = profit.std(ddof=1) / np.sqrt(len(profit))

    return pd.Series({
        'Expected Profit ($/acre)': profit.mean(),
        'Profit Std Dev ($/acre)': profit.std(ddof=1),
        'Probability Profit < 0': (profit < 0).mean(),
        'P5 Profit ($/acre)': p5,
        'CVaR 5% Profit ($/acre)': cvar5,
        'Probability Overhedged': group['overhedged'].mean(),
        'Probability Margin Call Proxy': group['margin_call_proxy'].mean(),
        'Avg Transaction Cost ($/acre)': group['transaction_cost_usd'].mean() / FARM_ACRES,
        'Avg Margin Financing Cost ($/acre)': group['margin_financing_cost_usd'].mean() / FARM_ACRES,
        'Mean 95% CI Low ($/acre)': profit.mean() - Z_95 * standard_error,
        'Mean 95% CI High ($/acre)': profit.mean() + Z_95 * standard_error,
    })

fixed_summary = (
    fixed_results.groupby('strategy', sort=False)
    .apply(summarize_strategy, include_groups=False)
    .reset_index()
)

print(fixed_summary.round(4).to_string(index=False))

## 10. 应用预先锁定的Eligibility Rules

In [ ]:
unhedged_mean = fixed_summary.loc[
    fixed_summary['strategy'] == 'Fixed 0%',
    'Expected Profit ($/acre)',
].iloc[0]

mean_profit_floor = (
    unhedged_mean
    - MAX_MEAN_PROFIT_SHORTFALL_FRACTION * abs(unhedged_mean)
)

fixed_summary['Pass Expected-Profit Rule'] = (
    fixed_summary['Expected Profit ($/acre)'] >= mean_profit_floor
)
fixed_summary['Pass Overhedge Rule'] = (
    fixed_summary['Probability Overhedged'] <= MAX_OVERHEDGE_PROBABILITY
)
fixed_summary['Eligible'] = (
    fixed_summary['Pass Expected-Profit Rule']
    & fixed_summary['Pass Overhedge Rule']
)

eligible_rows = fixed_summary[fixed_summary['Eligible']]
best_fixed_index = eligible_rows['CVaR 5% Profit ($/acre)'].idxmax()
fixed_summary['Provisional Preferred Fixed Strategy'] = False
fixed_summary.loc[best_fixed_index, 'Provisional Preferred Fixed Strategy'] = True

print(f'Unhedged expected profit = ${unhedged_mean:.4f}/acre')
print(f'Minimum allowed expected profit = ${mean_profit_floor:.4f}/acre')
print(fixed_summary[[
    'strategy',
    'Expected Profit ($/acre)',
    'CVaR 5% Profit ($/acre)',
    'Probability Overhedged',
    'Eligible',
    'Provisional Preferred Fixed Strategy',
]].round(4).to_string(index=False))

## 11. 为什么Fixed 100%没有被选中？

Fixed 100%在当前模拟中有最高expected profit和略低的standard deviation，但它有两个问题：

1. 由于合约按预计产量建立，而实际产量可能较低，over-hedging probability很高；
2. 它的CVaR 5%反而比Fixed 75%更差。

这说明“套保越多越安全”并不总成立。超过实际产量的空头会重新引入风险。

In [ ]:
fixed75 = fixed_summary[fixed_summary['strategy'] == 'Fixed 75%'].iloc[0]
fixed100 = fixed_summary[fixed_summary['strategy'] == 'Fixed 100%'].iloc[0]

print(f"Fixed 75% CVaR 5%: ${fixed75['CVaR 5% Profit ($/acre)']:.2f}/acre")
print(f"Fixed 100% CVaR 5%: ${fixed100['CVaR 5% Profit ($/acre)']:.2f}/acre")
print(f"Fixed 75% overhedge probability: {fixed75['Probability Overhedged']:.2%}")
print(f"Fixed 100% overhedge probability: {fixed100['Probability Overhedged']:.2%}")

## 12. 图1：Expected Profit、P5与CVaR 5%

In [ ]:
x = fixed_summary['strategy']

plt.figure(figsize=(10, 5))
plt.plot(x, fixed_summary['Expected Profit ($/acre)'], marker='o', label='Expected Profit')
plt.plot(x, fixed_summary['P5 Profit ($/acre)'], marker='o', label='P5')
plt.plot(x, fixed_summary['CVaR 5% Profit ($/acre)'], marker='o', label='CVaR 5%')
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('USD per acre')
plt.title('Fixed Hedge Ratios: Mean and Downside Profit')
plt.legend()
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 13. 图2：波动、Over-hedging与Margin-call Proxy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(
    fixed_summary['strategy'],
    fixed_summary['Profit Std Dev ($/acre)'],
    color='#4472C4',
)
axes[0].set_title('Profit Standard Deviation')
axes[0].set_ylabel('USD per acre')
axes[0].tick_params(axis='x', rotation=25)

xpos = np.arange(len(fixed_summary))
width = 0.36
axes[1].bar(
    xpos - width/2,
    fixed_summary['Probability Overhedged'],
    width,
    label='Overhedged',
    color='#ED7D31',
)
axes[1].bar(
    xpos + width/2,
    fixed_summary['Probability Margin Call Proxy'],
    width,
    label='Margin-call proxy',
    color='#A5A5A5',
)
axes[1].axhline(MAX_OVERHEDGE_PROBABILITY, color='#C00000', linestyle='--', label='10% overhedge limit')
axes[1].set_xticks(xpos, fixed_summary['strategy'], rotation=25)
axes[1].set_title('Implementation Risks')
axes[1].set_ylabel('Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. 必须通过的策略检查

In [ ]:
expected_contracts = {
    'Fixed 0%': 0,
    'Fixed 25%': 11,
    'Fixed 50%': 21,
    'Fixed 75%': 32,
    'Fixed 100%': 42,
}

assert len(fixed_results) == len(FIXED_HEDGE_RATIOS) * N_SIMULATIONS
assert not fixed_results.duplicated(['scenario_id', 'strategy']).any()
assert fixed_results.isna().sum().sum() == 0

for strategy, contracts in expected_contracts.items():
    group = fixed_results[fixed_results['strategy'] == strategy]
    assert (group['initial_contracts'] == contracts).all()
    assert (group['july_adjustment_contracts'] == 0).all()
    assert (group['final_contracts'] == contracts).all()

assert fixed_summary.loc[
    fixed_summary['strategy'] == 'Fixed 75%',
    'Provisional Preferred Fixed Strategy',
].iloc[0]

assert not fixed_summary.loc[
    fixed_summary['strategy'] == 'Fixed 100%',
    'Eligible',
].iloc[0]

print('全部策略检查通过。')

## 15. 可重复性检查

In [ ]:
expected_metrics = {
    'Fixed 0%':   [-1.5944606006, 153.7292722166, -308.6271876201, -338.7472297638, 0.0000, 0.0000],
    'Fixed 25%':  [ 5.4810561656, 113.6742245880, -223.6044596490, -240.0045877430, 0.0000, 0.1435],
    'Fixed 50%':  [11.9133441351,  80.0589484637, -144.9766714504, -154.8015607295, 0.0000, 0.3716],
    'Fixed 75%':  [18.9888609010,  52.8491026060,  -67.6507728180,  -94.9466253290, 0.0000, 0.4751],
    'Fixed 100%': [25.4211488710,  52.2506969540,  -71.8223184210, -118.1373768120, 0.4538, 0.5330],
}

metric_columns = [
    'Expected Profit ($/acre)',
    'Profit Std Dev ($/acre)',
    'P5 Profit ($/acre)',
    'CVaR 5% Profit ($/acre)',
    'Probability Overhedged',
    'Probability Margin Call Proxy',
]

for strategy, expected_values in expected_metrics.items():
    actual_values = fixed_summary.loc[
        fixed_summary['strategy'] == strategy,
        metric_columns,
    ].iloc[0].to_numpy(dtype=float)
    assert np.allclose(actual_values, expected_values, atol=1e-6), (
        strategy, actual_values, expected_values
    )

print('五种固定策略的可重复性检查通过。')

## 16. 保存第9课结果

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_09_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

results_path = OUTPUT_DIR / 'fixed_strategy_scenario_results_50000.csv'
summary_path = OUTPUT_DIR / 'fixed_strategy_summary.csv'
decision_path = OUTPUT_DIR / 'step_09_fixed_strategy_decision.json'

fixed_results.to_csv(results_path, index=False)
fixed_summary.to_csv(summary_path, index=False)

preferred_fixed = fixed_summary.loc[
    fixed_summary['Provisional Preferred Fixed Strategy'], 'strategy'
].iloc[0]

decision = {
    'scope': 'fixed strategies only',
    'selection_objective': 'maximize CVaR 5% profit among eligible strategies',
    'expected_profit_rule': 'mean profit cannot be more than 5% of abs(unhedged mean) below unhedged mean',
    'max_overhedge_probability': MAX_OVERHEDGE_PROBABILITY,
    'provisional_preferred_fixed_strategy': preferred_fixed,
}
decision_path.write_text(json.dumps(decision, indent=2), encoding='utf-8')

print('已保存:', results_path)
print('已保存:', summary_path)
print('已保存:', decision_path)
print('Fixed strategies中的暂时首选:', preferred_fixed)

## 17. 本课结论

在五种fixed strategies中：

- Fixed 100%有最高expected profit和最低standard deviation，但over-hedging probability约45.38%，超过10%限制，因此不合格；
- Fixed 75%的CVaR 5%约为−$94.95/acre，是合格fixed strategies中最高的；
- 因此Fixed 75%是**暂时最佳固定策略**。

Margin-call proxy随套保比例上升，Fixed 75%约为47.51%。这说明收获期利润风险降低，不代表途中不需要流动性。

不能把本课结论写成最终推荐，因为adaptive July strategy还没有加入。

**下一课：建立July adaptive strategy，根据July PDSI和更新后的产量预测，把目标比例调整为25%、50%或75%，并单独计算July adjustment P&L。**